# Masked-Diffusion BabyLM — Strict-Small (Training)

Train a LLaDA/MDLM-style masked-diffusion language model on the BabyLM 2026
**Strict-Small** corpus (≤10M words, ≤10 epochs, ≤100M words seen).

Everything heavy (data, tokenizer, runs) lives on Google Drive so a Colab
disconnect never loses work. Data prep is skipped if already done, and training
**auto-resumes** from the last checkpoint — re-run the training cell after any
disconnect (even on another day) and it continues where it left off. Run the
cells top to bottom.

**Before you start:** add an `HF_TOKEN` (read access) via the Colab Secrets panel (key icon).

In [ ]:
# Step 1 — Parameters (edit these)
REPO_URL   = "https://github.com/<your-user>/diffusion-babylm-eval.git"
DRIVE_ROOT = "/content/drive/MyDrive/BabyLM/diffusion"
CONDITION  = "MD_base"   # MD_base | MD_freq_mask | MD_layerdup
SEED       = 42

In [ ]:
# Step 2 — Mount Drive + clone/pull the repo
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
if not os.path.isdir("/content/diffusion-babylm-eval"):
    subprocess.run(["git", "clone", REPO_URL, "/content/diffusion-babylm-eval"], check=True)
%cd /content/diffusion-babylm-eval/diffusion
subprocess.run(["git", "pull"], check=False)

In [ ]:
# Step 3 — HuggingFace token (from Colab Secrets)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets (key icon).", e)

In [ ]:
# Step 4 — Bootstrap: install + Drive symlinks + data + smoke test
!bash scripts/colab_bootstrap.sh --drive-root "$DRIVE_ROOT"

In [ ]:
# Step 5 — Train. Checkpoints (model + optimizer state) are written to
# runs/<run>/checkpoints/ on Drive. If Colab disconnects, just RE-RUN this cell:
# it auto-detects the latest checkpoint for this (condition, seed) and CONTINUES
# from there — even on a different day. Use --no-resume to force a fresh run.
!python scripts/train.py --condition $CONDITION --seed $SEED -v \
    --token-data data/tokens --tokenizer tokenizer/mdlm_bpe_16k

In [ ]:
# Step 6 — Quick training-loss curve from the run log
import glob, json
import matplotlib.pyplot as plt

run = sorted(glob.glob(f"runs/*_{CONDITION}_seed{SEED}"))[-1]
steps, train, eval_steps, val = [], [], [], []
for line in open(f"{run}/log.jsonl"):
    r = json.loads(line)
    if r["phase"] == "train":
        steps.append(r["step"]); train.append(r["avg_train_loss"])
    else:
        eval_steps.append(r["step"]); val.append(r["val_loss"])
plt.figure(figsize=(8, 4))
plt.plot(steps, train, label="train")
plt.plot(eval_steps, val, "o-", label="val")
plt.xlabel("step"); plt.ylabel("MDLM loss"); plt.legend(); plt.title(run.split('/')[-1])
plt.show()